In [1]:
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password="password",
    host="localhost",
    port=5555,
    database="similarity_search_service_db"
)

In [2]:
from typing import List
from pgvector.sqlalchemy import Vector
from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


# Create the base class for the table definition
class Base(DeclarativeBase):
    __abstract__ = True


# Create the table definition
class Images(Base):
    __tablename__ = "images"
    VECTOR_LENGTH = 512
    
    # primary key
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    # image path - we will use it to store the path to the image file, after similarity search we can use it to retrieve the image and display it
    image_path: Mapped[str] = mapped_column(String(256))
    # image embedding - we will store the image embedding in this column, the image embedding is a list of 512 floats this is the output of the sentence transformer model
    image_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

In [3]:
from sqlalchemy import create_engine

engine = create_engine(db_url)

In [4]:
Base.metadata.create_all(engine)

In [5]:
from sqlalchemy.orm import Session
from sqlalchemy import Engine, select
import numpy as np

# reusable function to insert data into the table
def insert_image(engine: Engine, image_path: str, image_embedding: list[float]):
    with Session(engine) as session:
        # create the image object
        image = Images(image_path=image_path, image_embedding=image_embedding)
        # add the image object to the session
        session.add(image)
        # commit the transaction
        session.commit()

# insert some data into the table
N = 100
for i in range(N):
    image_path = f"image_{i}.jpg"
    image_embedding = np.random.rand(512).tolist()
    insert_image(engine, image_path, image_embedding)

# select first image from the table
with Session(engine) as session:
    image = session.query(Images).first()


# Note: pgvector exposes *distances*, not similarities (lower = more similar).
# To rank, sort ascending by `cosine_distance`. To filter by a similarity threshold,
# use `1 - cosine_distance > threshold`.

# order the K images by ascending cosine distance to the first image (smallest distance = most similar)
def find_k_images(engine: Engine, k: int, original_image: Images) -> list[Images]:
    with Session(engine) as session:
        # execution_options={"prebuffer_rows": True} is used to prebuffer the rows, this is useful when we want to fetch the rows in chunks and return them after session is closed
        result = session.execute(
            select(Images)
            .order_by(Images.image_embedding.cosine_distance(original_image.image_embedding))
            .limit(k),
            execution_options={"prebuffer_rows": True}
        ).scalars().all()
        return result

# find the 10 most similar images to the first image
k = 10
similar_images = find_k_images(engine, k, image)

In [6]:
# find images with similarity to the original above the threshold (similarity = 1 - cosine_distance)
def find_images_with_similarity_score_greater_than(engine: Engine, similarity_score: float, original_image: Images) -> list[Images]:
    with Session(engine) as session:
        result = session.execute(
            select(Images)
            .filter((1 - Images.image_embedding.cosine_distance(original_image.image_embedding)) > similarity_score),
            execution_options={"prebuffer_rows": True}
        ).scalars().all()
        return result

In [7]:
from datasets import load_dataset

dataset = load_dataset("FronkonGames/steam-games-dataset")

# get columns names and types
columns = dataset["train"].features
print(columns)

columns_to_keep = ["name", "windows", "linux", "mac", "detailed_description", "supported_languages", "price"]

N = 40000 # you can adjust this number
dataset = dataset["train"].select_columns(columns_to_keep).select(range(N))

README.md:   0%|          | 0.00/596 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/124146 [00:00<?, ? examples/s]

{'appID': Value('string'), 'name': Value('string'), 'release_date': Value('string'), 'estimated_owners': Value('string'), 'peak_ccu': Value('int64'), 'required_age': Value('int64'), 'price': Value('float64'), 'dlc_count': Value('int64'), 'detailed_description': Value('string'), 'short_description': Value('string'), 'supported_languages': List(Value('string')), 'full_audio_languages': List(Value('string')), 'reviews': Value('string'), 'header_image': Value('string'), 'website': Value('string'), 'support_url': Value('string'), 'support_email': Value('string'), 'windows': Value('bool'), 'mac': Value('bool'), 'linux': Value('bool'), 'metacritic_score': Value('int64'), 'metacritic_url': Value('string'), 'user_score': Value('int64'), 'positive': Value('int64'), 'negative': Value('int64'), 'score_rank': Value('string'), 'achievements': Value('int64'), 'recommendations': Value('int64'), 'notes': Value('string'), 'average_playtime_forever': Value('int64'), 'average_playtime_2weeks': Value('int6

In [8]:
from sqlalchemy import Integer, Float, Boolean, Text


class Games(Base):
    __tablename__ = "games"
    __table_args__ = {'extend_existing': True}
    
    # the vector size produced by the model taken from documentation https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2
    VECTOR_LENGTH = 512
        
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(256))
    # `Text` is an unbounded string in Postgres — game descriptions can run a few KB
    description: Mapped[str] = mapped_column(Text)
    windows: Mapped[bool] = mapped_column(Boolean)
    linux: Mapped[bool] = mapped_column(Boolean)
    mac: Mapped[bool] = mapped_column(Boolean)
    price: Mapped[float] = mapped_column(Float)
    game_description_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

In [9]:
from sentence_transformers import SentenceTransformer


checkpoint = "distiluse-base-multilingual-cased-v2"
model = SentenceTransformer(checkpoint)


def generate_embeddings(text: str) -> list[float]:
    return model.encode(text)

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

In [10]:
from tqdm import tqdm


def insert_games(engine, dataset):
    with tqdm(total=len(dataset)) as pbar:
       for i, game in enumerate(dataset):
           game_description = game["detailed_description"] or ""
           game_embedding = generate_embeddings(game_description)
           name, windows, linux, mac, price = game["name"], game["windows"], game["linux"], game["mac"], game["price"]
           # keep rows with the required fields; windows/linux/mac are platform flags (True/False),
           # not missing-data signals — don't gate the insert on them or you discard most of the dataset
           if name and game_description and price is not None:
               game = Games(
                   name=game["name"], 
                   description=game_description,
                   windows=game["windows"], 
                   linux=game["linux"], 
                   mac=game["mac"], 
                   price=game["price"], 
                   game_description_embedding=game_embedding
               )
               with Session(engine) as session:
                   session.add(game)
                   session.commit()
           pbar.update(1)

In [11]:
insert_games(engine, dataset)

100%|██████████| 40000/40000 [03:20<00:00, 199.57it/s]


In [14]:
from typing import Optional

def find_game(
    engine: Engine, 
    game_description: str, 
    windows: Optional[bool] = None, 
    linux: Optional[bool] = None,
    mac: Optional[bool] = None,
    price: Optional[int] = None
):
    with Session(engine) as session:
        game_embedding = generate_embeddings(game_description)
    
        query = (
            select(Games)
            .order_by(Games.game_description_embedding.cosine_distance(game_embedding))
        )
        
        if price:
            query = query.filter(Games.price <= price)
        if windows:
            query = query.filter(Games.windows == True)
        if linux:
            query = query.filter(Games.linux == True)
        if mac:
            query = query.filter(Games.mac == True)
        
        result = session.execute(query, execution_options={"prebuffer_rows": True})
        game = result.scalars().first()
        
        return game

In [15]:
game = find_game(engine, "This is a game about a hero who saves the world", price=10)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

game = find_game(engine, game_description="Home decorating", price=20)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: Hero's everyday life
Description: In Hero's everyday life, you need to play the role of a hero, completing a variety of tasks from give and bring, to kill monsters and bosses. Help all the different game characters with their mundane tasks and become famous throughout the fantasy kingdom as a great hero! Or just fool around and have as much fun as you can! The world is filled with various secrets, there are also enemies who are ready to break you no matter what, so arm yourself with a sword and hit the road. Who is a hero? Is this the one who gets up in the morning for an adventure? Or the one who fights for the good and prosperity? In general, you will need to decide which of you will be the hero and for what you will fight! The kingdom of fantasy is filled with monsters, NPS cannot fulfill their mission and therefore they called you a hero, and whether you are a great hero or not, it's up to you to decide! Complete a mission, fight monsters, kill bosses, or just insult the king

In [16]:
game = find_game(engine, game_description="Home decorating", mac=True, price=5)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: My Universe - Interior Designer
Description: Paint colours, furnishings, style, interior design, you choose each and every detail. Talk to your clients to find out more about the hopes, dreams and preferences they have for their homes. You're going to have to renovate their houses and complete the mission objectives to satisfy their needs. The happier they are with your work, the more your notoriety and level will increase! Gradually, you'll unlock new items, styles, decorative possibilities and bigger projects. During your adventure, you can play in 2 different modes: renovation mode where you can change the structure of the house by knocking down or putting up walls, taking up wooden or tiled flooring, painting, changing the wall coverings, fixing the electrics or adding new water inlets. You can also switch to decoration mode where you can choose and position the furnishings, plants, rugs, wallpaper, curtains, etc. It's up to you to add real character to the house. And if you 

## Miluvs

In [17]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(
    host=host,
    port=port
)

In [18]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")

In [19]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding", 
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
) 

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 470143606894104970, 'consistency_level': 2, 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 470143785379037201, 'update_timestamp': 470143785379037201}


In [20]:
# define data source and destination
## the document origin destination from which document will be downloaded 
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document 
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [21]:
# download data
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)

In [22]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)

In [23]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))
        
        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [26]:
# search

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings", 
        data=[embedded_query], 
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [27]:
result

data: [[{'id': 470143606894105310, 'distance': 29.125164031982422, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mog

## OpenRouter API integration

Instead of the Gemini API, I used OpenRouter, because I already had an API key for it and it shouldn't run into the rate limits.

In [ ]:
import os
from dotenv import load_dotenv
from openrouter import OpenRouter

load_dotenv()

OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")
openrouter_client = OpenRouter(api_key=OPENROUTER_KEY)

MODEL = "google/gemini-2.5-flash"

def generate_response(prompt: str):
    try:
        response = openrouter_client.chat.send(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [ ]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""Answer the question using only the context below.
    
Context:
{context}

Question: {query}

Answer:"""
    return prompt
    

def rag(model, query: str) -> str:
    result = search(model, query)
    context = "\n\n".join(hit["entity"]["text"] for hit in result[0])
    prompt = build_prompt(context, query)
    return generate_response(prompt)


print(rag(model, "Czym jest sztuczna inteligencja?"))

Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań, które wymagają ludzkiej inteligencji.


In [ ]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""Answer the question using only the context below. Always answer in Japanese even though the context is in Polish.

Context:
{context}

Question: {query}

Answer:"""
    return prompt


print(rag(model, "Czym jest sztuczna inteligencja?"))

人工知能（AI）は、人間の知能を必要とするタスクを実行できるコンピュータープログラムの作成に焦点を当てたコンピューターサイエンスの分野です。これらのタスクには、パターン認識、自然言語理解、意思決定、学習、計画などが含まれます。AIの主な目的は、人間のような方法で考え、意思決定できるシステムを作成することです。口語的には、「人工」とは、自然なものを模倣するように作られた製品を意味します。このような意味で、「人工」という言葉は、人工スケートリンクや人工眼について話すときに使われます。人工知能は、自然な人間の知能をシミュレートするもの（プログラム、機械）です。


In [ ]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""Answer the question using only the context below. Answer in exactly three short bullet points.

Context:
{context}

Question: {query}

Answer:"""
    return prompt


print(rag(model, "Czym jest sztuczna inteligencja?"))

* Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań, które wymagają ludzkiej inteligencji.
* Głównym celem AI jest stworzenie systemów, które są zdolne do myślenia i podejmowania decyzji na sposób przypominający ludzki.
* W języku potocznym „sztuczny" oznacza to, co jest wytworem mającym naśladować coś naturalnego, a sztuczna inteligencja byłaby czymś symulującym inteligencję naturalną, ludzką.


In [ ]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""Answer the question using only the context below in Polish. Use slangs and pirate language.

Context:
{context}

Question: {query}

Answer:"""
    return prompt


print(rag(model, "Czym jest sztuczna inteligencja?"))

Ahoj, szczury lądowe! Sztuczna inteligencja, to tam, to coś, co symuluje naturalną, ludzką inteligencję, jak te sztuczne lodowiska czy oka! To po prostu taki kawał informatyki, co tworzy programy, by one umiały robić zadania, co to ludzka głowa musi! Rozpoznawanie wzorców, gadanie w ludzkim języku, decyzje podejmować, uczyć się, planować – wszystko to! Ich główny cel to stworzyć systemy, co myślą i decydują, jak my, ludzie!

A pamiętajcie, że to bydlę, Deep Blue, w 1997 roku zatopiło samego Kasparowa w szachach! Arrr!
